Instalowanie bibliotek

In [ ]:
!pip install -U scikit-learn==1.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 116.7 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2


In [ ]:
!pip install -U umap-learn
!pip install -U mlxtend
!pip install -U imbalanced-learn

In [ ]:
!pip install --quiet joblib pandas matplotlib seaborn
!pip install --quiet xgboost

Zaimportowanie moduły i połączenie SQLite

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set(style="whitegrid")

db_path = "/content/database.sqlite"

Łączymy się i wyświetlamy listę tabel (aby mieć pewność, że plik został odczytany)

In [ ]:
conn = sqlite3.connect(db_path)
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tables in DB:\n", tables)

Tables in DB:
                 name
0    sqlite_sequence
1  Player_Attributes
2             Player
3              Match
4             League
5            Country
6               Team
7    Team_Attributes


Pobierzmy główną tabelę mecze i przyjrzyjmy się jej.

In [ ]:
df_matches = pd.read_sql_query("SELECT * FROM Match", conn)
print("Rows:", len(df_matches))
df_matches.head(5)

Rows: 25979


,id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,SJA,VCH,VCD,VCA,GBH,GBD,GBA,BSH,BSD,BSA
0,1,1,1,2008/2009,1,2008-08-17 00:00:00,492473,9987,9993,1,...,4.00,1.65,3.40,4.50,1.78,3.25,4.00,1.73,3.40,4.20
1,2,1,1,2008/2009,1,2008-08-16 00:00:00,492474,10000,9994,0,...,3.80,2.00,3.25,3.25,1.85,3.25,3.75,1.91,3.25,3.60
2,3,1,1,2008/2009,1,2008-08-16 00:00:00,492475,9984,8635,0,...,2.50,2.35,3.25,2.65,2.50,3.20,2.50,2.30,3.20,2.75
3,4,1,1,2008/2009,1,2008-08-17 00:00:00,492476,9991,9998,5,...,7.50,1.45,3.75,6.50,1.50,3.75,5.50,1.44,3.75,6.50
4,5,1,1,2008/2009,1,2008-08-16 00:00:00,492477,7947,9985,1,...,1.73,4.50,3.40,1.65,4.50,3.50,1.65,4.75,3.30,1.67


Wybieranie i czyszczenie kolumn, tworzenie etykiety docelowej

Weźmiemy: date, home_team_api_id, away_team_api_id, home_team_goal, away_team_goal

Stwórzmy: result:

0 → home win

1 → draw

2 → away win

In [ ]:
# Upewnijmy się, że wymagane kolumny są
cols_needed = ['date', 'home_team_api_id', 'away_team_api_id', 'home_team_goal', 'away_team_goal']
for c in cols_needed:
    if c not in df_matches.columns:
        raise ValueError(f"Column {c} not found in DB. Check table schema.")

# Bierzemy tylko niezbędne kolumny i wprowadzamy datę datetime
df = df_matches[cols_needed].copy()
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
df['date'] = pd.to_datetime(df['date'])

# utwórz etykietę docelową
def get_result(row):
    if row['home_team_goal'] > row['away_team_goal']:
        return 0  # home win
    elif row['home_team_goal'] == row['away_team_goal']:
        return 1  # draw
    else:
        return 2  # away win

df['result'] = df.apply(get_result, axis=1)

# sortujemy według daty (rosnąco)
df = df.sort_values('date').reset_index(drop=True)
print("Total matches:", df.shape[0])
df.head()

Total matches: 25979


,date,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,result
0,2008-07-18,10192,9931,1,2,2
1,2008-07-19,9930,10179,3,1,0
2,2008-07-20,10199,9824,1,2,2
3,2008-07-20,7955,10243,1,2,2
4,2008-07-23,6493,7955,1,2,2


Feature engineering — Elo rating + recent goals averages

Wskaźnik ELO dla każdej drużyny (zainicjowany do 1500 i aktualizowany przed każdym meczem)

Elo_diff = elo_home - elo_away

Przewaga własnego boiska: binarna (1 = gospodarze)

Średnia liczba bramek strzelonych przez drużynę gospodarzy w ostatnich N meczach (N=5)

Średnia liczba bramek strzelonych przez drużynę gości w ostatnich N meczach (N=5)

In [ ]:
# parametry
INIT_ELO = 1500
K = 20         # wskaźnik uczenia się dla ELO
N_RECENT = 5   # liczba ostatnich meczów ze średnią liczbą goli

# Słowniki do przechowywania informacji
elo = {}  # team_id -> current elo
recent_goals_for = {}  # team_id -> list of last goals scored
recent_goals_against = {}  # team_id -> list of last goals conceded

# funkcje pomocnicze
def get_elo(team_id):
    return elo.get(team_id, INIT_ELO)

def update_recent(team_id, goals_for, goals_against):
    recent_goals_for.setdefault(team_id, []).append(goals_for)
    recent_goals_against.setdefault(team_id, []).append(goals_against)
    # trim lists to N_RECENT
    if len(recent_goals_for[team_id]) > N_RECENT:
        recent_goals_for[team_id].pop(0)
    if len(recent_goals_against[team_id]) > N_RECENT:
        recent_goals_against[team_id].pop(0)

def avg_last(lst):
    return np.mean(lst) if len(lst)>0 else np.nan

# przygotujmy kolumny
df['elo_home'] = np.nan
df['elo_away'] = np.nan
df['elo_diff'] = np.nan
df['home_recent_goals_avg'] = np.nan
df['away_recent_goals_avg'] = np.nan
df['home_recent_conceded_avg'] = np.nan
df['away_recent_conceded_avg'] = np.nan

for idx, row in df.iterrows():
    h = row['home_team_api_id']
    a = row['away_team_api_id']
    # aktualny ELO przed meczem
    eh = get_elo(h)
    ea = get_elo(a)
    df.at[idx, 'elo_home'] = eh
    df.at[idx, 'elo_away'] = ea
    df.at[idx, 'elo_diff'] = eh - ea
    # recent goals averages
    df.at[idx, 'home_recent_goals_avg'] = avg_last(recent_goals_for.get(h, []))
    df.at[idx, 'home_recent_conceded_avg'] = avg_last(recent_goals_against.get(h, []))
    df.at[idx, 'away_recent_goals_avg'] = avg_last(recent_goals_for.get(a, []))
    df.at[idx, 'away_recent_conceded_avg'] = avg_last(recent_goals_against.get(a, []))
    # wynik meczu
    g_h = row['home_team_goal']
    g_a = row['away_team_goal']
    # expected score for home
    expected_h = 1 / (1 + 10 ** ((ea - eh) / 400.0))
    expected_a = 1 - expected_h
    # actual score in Elo terms: win=1, draw=0.5, loss=0
    if g_h > g_a:
        s_h = 1.0
        s_a = 0.0
    elif g_h == g_a:
        s_h = 0.5
        s_a = 0.5
    else:
        s_h = 0.0
        s_a = 1.0
    # update Elo
    new_eh = eh + K * (s_h - expected_h)
    new_ea = ea + K * (s_a - expected_a)
    elo[h] = new_eh
    elo[a] = new_ea
    # update recent goals lists
    update_recent(h, g_h, g_a)
    update_recent(a, g_a, g_h)

# Sprawdźmy luki – pierwsze kilka meczów nie ma aktualnych średnich, wypełnimy je zerami.
df[['elo_home','elo_away','elo_diff','home_recent_goals_avg','away_recent_goals_avg']].head(10)

df['home_recent_goals_avg'] = df['home_recent_goals_avg'].fillna(0)
df['away_recent_goals_avg'] = df['away_recent_goals_avg'].fillna(0)
df['home_recent_conceded_avg'] = df['home_recent_conceded_avg'].fillna(0)
df['away_recent_conceded_avg'] = df['away_recent_conceded_avg'].fillna(0)

# stwórzmy X i Y
feature_cols = [
    'elo_home', 'elo_away', 'elo_diff',
    'home_recent_goals_avg', 'home_recent_conceded_avg',
    'away_recent_goals_avg', 'away_recent_conceded_avg'
]
X = df[feature_cols].astype(float)
y = df['result'].astype(int)

print("X shape:", X.shape, "y distribution:\n", y.value_counts())

X shape: (25979, 7) y distribution:
 result
0    11917
2     7466
1     6596
Name: count, dtype: int64


Train / Test split

Train — wczesne sezony, test — późne sezony

In [ ]:
# podział dat: train = wszystkie mecze przed 2016-01-01, test = później
split_date = pd.to_datetime("2016-01-01")
train_idx = df['date'] < split_date
test_idx = df['date'] >= split_date

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])

Train size: 24349 Test size: 1630


edukacja na XGBoost

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Tworzenie modelu XGBoost
model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

# Szkolimy na już podzielonych danych
model.fit(X_train, y_train)

# Przewidywanie i ocena
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.5067484662576687
Confusion Matrix:
 [[596  12 113]
 [302  11  98]
 [268  11 219]]
Classification Report:
               precision    recall  f1-score   support

           0       0.51      0.83      0.63       721
           1       0.32      0.03      0.05       411
           2       0.51      0.44      0.47       498

    accuracy                           0.51      1630
   macro avg       0.45      0.43      0.38      1630
weighted avg       0.46      0.51      0.44      1630



In [ ]:
# Zapisywanie modelu w formacie JSON
model.save_model("xgb_model.json")

In [ ]:
from google.colab import files
files.download("xgb_model.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip freeze > requirements.txt